# Lekcija 25: Spajanje slika i izrada mozaika

Ova lekcija povezuje detekciju i podudaranje značajki (lekcija 20), robusnu prilagodbu (lekcija 23) i procjenu homografije (lekcija 24) u cjelovit postupak koji spaja dvije preklapajuće fotografije u jednu panoramu bez vidljivog spoja.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Sažeti postupak

1. **Detektirati i podudariti značajke** između dviju fotografija (lekcija 20: SIFT + test omjera).
2. **Robusno procijeniti homografiju** koja povezuje ravninu jedne slike s drugom (homografija iz lekcije 24, RANSAC iz lekcije 23).
3. **Preslikati** jednu sliku u koordinatni sustav druge (lekcija 9: `cv2.warpPerspective`).
4. **Sastaviti i pomiješati** dvije slike na jednom platnu, uz postupni prijelaz preko preklopa kako spoj ne bi bio vidljiv.

## Dvije preklapajuće fotografije

Dvije fotografije iste zgrade od opeke, snimljene iz malo različitih položaja uz znatno preklapanje.

In [ ]:
view1 = cv2.imread('../img/clemson00.jpg')
view2 = cv2.imread('../img/clemson01.jpg')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(view1, cv2.COLOR_BGR2RGB))
axes[0].set_title('View 1')
axes[1].imshow(cv2.cvtColor(view2, cv2.COLOR_BGR2RGB))
axes[1].set_title('View 2')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Izvor slike: Stan Birchfield</p>

## Korak 1: podudaranje značajki

In [ ]:
gray1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

print(f'keypoints: {len(kp1)} (view 1), {len(kp2)} (view 2)')
print(f'good matches after ratio test: {len(good_matches)}')

match_vis = cv2.drawMatches(view1, kp1, view2, kp2, good_matches[:60], None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(11, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title('Feature matches in the overlap region (first 60 shown)')
plt.axis('off')
plt.show()

## Korak 2: robusna homografija

Određujemo homografiju koja preslikava točke pogleda 2 u koordinatni sustav pogleda 1, kako bi preslikavanje pogleda 2 njome završilo na odgovarajućem mjestu zajedničkog platna.

In [ ]:
pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

H, inlier_mask = cv2.findHomography(pts2, pts1, cv2.RANSAC, 3.0)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered homography:')
print(np.round(H, 4))

## Koraci 3–4: preslikavanje, sastavljanje i miješanje

Najprije određujemo potrebnu veličinu izlaznog platna, preslikavanjem kutova obiju slika pomoću `H` (lekcija 24) i uzimanjem graničnog pravokutnika &mdash; za razliku od sintetičkog slučaja, ove fotografije nisu jednostavni izrezi zajedničkog kadra poravnani s osima, pa treba izračunati i veličinu platna i pomak pogleda 1 unutar njega umjesto da ih pretpostavimo. Preslikavamo pogled 2 na to platno, a zatim miješamo područje preklopa jednostavnim **postupnim prijelazom (feathering)**: linearnom promjenom alfe od „potpuno pogled 1” do „potpuno pogled 2” preko preklopa, istim miješanjem ponderiranim zbrojem kao u `cv2.addWeighted` iz lekcije 2, samo s prostorno promjenjivom težinom umjesto stalne.

In [ ]:
h1, w1 = view1.shape[:2]
h2, w2 = view2.shape[:2]
corners1 = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
corners2 = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]]).reshape(-1, 1, 2)
warped_corners2 = cv2.perspectiveTransform(corners2, H)
all_corners = np.concatenate([corners1, warped_corners2], axis=0)

x_min, y_min = np.floor(all_corners.min(axis=0).ravel()).astype(int)
x_max, y_max = np.ceil(all_corners.max(axis=0).ravel()).astype(int)
canvas_w, canvas_h = x_max - x_min, y_max - y_min

translation = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float64)
canvas1 = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas1[-y_min:-y_min + h1, -x_min:-x_min + w1] = view1
warped2 = cv2.warpPerspective(view2, translation @ H, (canvas_w, canvas_h))

has1 = canvas1.sum(axis=2) > 0
has2 = warped2.sum(axis=2) > 0
overlap = has1 & has2

overlap_cols = np.where(overlap.any(axis=0))[0]
x_start, x_end = overlap_cols.min(), overlap_cols.max()
ramp = np.clip((np.arange(canvas_w) - x_start) / (x_end - x_start + 1e-6), 0, 1)

alpha = np.zeros((canvas_h, canvas_w), dtype=np.float32)
alpha[overlap] = np.broadcast_to(ramp, (canvas_h, canvas_w))[overlap]

stitched = canvas1.astype(np.float32) * (1 - alpha[..., None]) + warped2.astype(np.float32) * alpha[..., None]
stitched[has1 & ~has2] = canvas1[has1 & ~has2]   # regions covered only by view 1
stitched[has2 & ~has1] = warped2[has2 & ~has1]   # regions covered only by view 2
stitched = stitched.astype(np.uint8)

plt.figure(figsize=(10, 4))
plt.imshow(cv2.cvtColor(stitched, cv2.COLOR_BGR2RGB))
plt.title('Stitched panorama')
plt.axis('off')
plt.show()

Krov (gore, potpuno vidljiv samo u pogledu 2) i grm (dolje, potpuno vidljiv samo u pogledu 1) oba se pojavljuju u mozaiku &mdash; potvrđujući da sastavljena slika doista koristi obje izvorne slike.

### Koliko je prilagodba dobra?

Za stvarnu fotografiju nemamo sintetičke referentne vrijednosti za usporedbu, ali nisu nam ni potrebne: same točke koje prolaze RANSAC daju izravnu mjeru geometrijske prilagodbe. Za svaku pripadnu korespondenciju projiciramo točku iz pogleda 2 pomoću `H` i mjerimo udaljenost do njezine odgovarajuće točke u pogledu 1 &mdash; **pogrešku reprojekcije**. Mala, usko grupirana pogreška reprojekcije znači da `H` dobro objašnjava geometriju pripadnih točaka.

In [ ]:
mask = inlier_mask.ravel().astype(bool)
pts2_h = np.hstack([pts2[mask], np.ones((mask.sum(), 1))])
proj = (H @ pts2_h.T).T
proj = proj[:, :2] / proj[:, 2:3]
reproj_error = np.linalg.norm(proj - pts1[mask], axis=1)

print(f'mean reprojection error: {reproj_error.mean():.3f} px')
print(f'max reprojection error:  {reproj_error.max():.3f} px')

plt.figure(figsize=(6, 3.5))
plt.hist(reproj_error, bins=20)
plt.xlabel('reprojection error (px)')
plt.ylabel('inlier count')
plt.title('Reprojection error of RANSAC inliers')
plt.tight_layout()
plt.show()

Pogreška reprojekcije manja od jednog piksela, znatno ispod RANSAC-ova praga pripadnosti od 3 px, potvrđuje da je `H` dobra geometrijska prilagodba &mdash; u skladu s urednim spojem vidljivim na gore spojenoj panorami.

## U praksi: `cv2.Stitcher`

OpenCV objedinjuje cijeli ovaj postupak (uz robusnije miješanje, kompenzaciju ekspozicije i podršku za više slika odjednom u bilo kojem rasporedu) u jednom pozivu visoke razine. Zadani način `PANORAMA` pretpostavlja slike iz kamere koja rotira oko svog optičkog središta i prije sastavljanja preslikava svaku sliku na sferu &mdash; izvrsno za široke panorame, ali savija ravne linije kada prizorom dominira ravna, pravocrtna struktura (poput pročelja zgrade), a kamera se translacijski pomaknula umjesto da je samo rotirala. Način `SCANS` umjesto toga izravno sastavlja slike pomoću ravninskih homografija, u skladu s gore korištenim pristupom.

In [ ]:
stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
status, panorama = stitcher.stitch([view1, view2])

print('status:', 'OK' if status == cv2.Stitcher_OK else f'failed ({status})')
if status == cv2.Stitcher_OK:
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    plt.title('cv2.Stitcher result (SCANS mode)')
    plt.axis('off')
    plt.show()

### Zadaci

1. Smanjite preklapanje dvaju pogleda (npr. prije podudaranja izrežite `view2` na njegovu krajnju desnu četvrtinu). Kada SIFT podudaranje pronalazi premalo dobrih podudaranja da bi `findHomography` dao pouzdan rezultat?
2. Zamijenite linearni postupni prijelaz oštrim rezom (bez miješanja: samo odaberite sliku koja pokriva svaki piksel, dijeleći preklop po sredini) i usporedite vidljivost spoja s varijantom postupnog prijelaza.
3. `H` je ovdje blaga opća homografija, a ne čista translacija, jer su dvije fotografije snimljene iz malo različitih položaja umjesto čistim bočnim pomakom. Ispišite omjer posljednjeg retka `H` prema `[0, 0, 1]` kao grubu mjeru obuhvaćene perspektivne deformacije i usporedite ga s rezultatom nametanja afine prilagodbe (`cv2.estimateAffinePartial2D`) &mdash; koliko dobro afina aproksimacija spaja slike u usporedbi s punom homografijom?